# CNN Training

Train a CNN network to extract the needle position of an analog pointer

### Basic Parameter

IMPORTANT: Do not rename any variables in this section — they are externally referenced in the GitHub action `Train Model`.

* `TFlite_MainType`: Model type name
* `TFlite_Version`: Model version identifier
* `TFlite_Size`: Model architecture defined by size
* `Validation_Percentage`: Fraction of images reserved for validation dataset
* `Input_Dir`: Path to the input directory containing training images
* `Output_Dir`: Path to the output directory where results (models, logs, etc.) will be saved
* `Input_Shape`: Image dimensions (width, height, channels)

In [ ]:
# Model type (No need to adapt)
TFlite_MainType: str = 'ana-class100'

# Define model version (4 characters, e.g. 0180 -> v1.80)
TFlite_Version: str  = 'xxxx'

# Choose model size
TFlite_Size: str     = 's1'
# Model size pool (Available models: `./src/models/ana_class100.py`)
Available_Model_Sizes = {'s1', 's2'}

#################################################################
# Validation dataset size
# Note: 0.0 = 0% validation size, use all images for training
Validation_Percentage = 0.2

# Folders
Input_Dir: str  = 'data_resize_all'
Output_Dir: str = 'models/ana-class100'

# Input image size [x, y, channels]
Input_Shape = (32, 32, 3)


### Load Libraries

In [ ]:
import os
import sys
import glob
from pathlib import Path
import random
import math
import numpy as np
import pandas as pd

from PIL import Image

import tensorflow as tf

from tensorflow import keras
from src.models.ana_class100 import *
from src.utils.augmentation import random_invert_image, random_white_balance


from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping
from tensorflow.keras.preprocessing.image import ImageDataGenerator

from sklearn.utils import shuffle
from sklearn.model_selection import train_test_split

import matplotlib.pyplot as plt
from src.utils.plot_functions import plot_dataset_analog, plot_dataset_analog_result, plot_divergence


%matplotlib inline
np.set_printoptions(precision=4)
np.set_printoptions(suppress=True)


# Make sure version is 4 characters long if version is defined as int (e.g. papermill paramter 100 -> 0100)
if isinstance(TFlite_Version, int):
    TFlite_Version = str(TFlite_Version).zfill(4)


# Validate model size input
if TFlite_Size not in Available_Model_Sizes:
    raise ValueError(f"Invalid TFlite_Size '{TFlite_Size}'. Must be one of: {', '.join(Available_Model_Sizes)}")


# Prepare folders
if not (Path(Input_Dir).exists() and Path(Input_Dir).is_dir()): # Check if input is availabe
    sys.exit(f"Folder '{Input_Dir}' does not exist.")
    
Path(Output_Dir).mkdir(parents=True, exist_ok=True)  # Create output folder if it doesn't exist


# Disable GPUs
try:
    tf.config.set_visible_devices([], 'GPU')
    visible_devices = tf.config.get_visible_devices()
    for device in visible_devices:
        assert device.device_type != 'GPU'
except:
    # Invalid device or cannot modify virtual devices once initialized.
    pass


### Load images

* The images are expected in the "Input_Dir"
* The image size must be 32 x 32 with 3 color channels (RGB)
* The first 3 digits of image filename must contain the real value representation of the image:
  * Generic: `x.y_zzzz.jpg`
  * Example: `4.6_main_ana1_2019-06-02T050011.jpg`

| Filename Part | Description                  | Usage                    |
|---------------|------------------------------|--------------------------|
| x.y           | Represented value (e.g. 4.6) | **Value to be learned**  |
| _zzzz         | Further file description     | Not required / processed |

* The images are stored in the x_data[]
* The expected output for each image in the corresponding y_data[]
* The last step is a shuffle (from sklearn.utils) as the filenames are on order due to the encoding of the expected analog readout in the filename
* Split dataset into training and validation parts

In [ ]:
# Load files
files = glob.glob(f"{Input_Dir}/*.jpg")
num_files = len(files)

# Prepare data container
f_data = np.empty(num_files, dtype="<U250")
x_data = np.empty((num_files, Input_Shape[0], Input_Shape[1], Input_Shape[2]), dtype="float32")
y_data = np.empty(num_files)

# Process files
for i, file in enumerate(files):
    image = Image.open(file)
    image = np.array(image, dtype="float32") # No resizing, use already resized images

    # Extract real value from filename and save as category [0.0 .. 9.9]
    base = Path(file).name
    category = float(base[:3])
    
    # Save data
    f_data[i] = file
    x_data[i] = image
    y_data[i] = category

# Shuffle data
x_data, y_data, f_data = shuffle(x_data, y_data, f_data)

# Split data to train and validation data
x_train, x_test, y_train, y_test = train_test_split(x_data, y_data, test_size=Validation_Percentage)
y_train = keras.utils.to_categorical(y_train * 10, 100)
y_test = keras.utils.to_categorical(y_test * 10, 100)


print("Data count: ", len(y_data))
print(x_data.shape)
print(np.expand_dims(y_data, axis=1).shape)


### Distribution of the data

Uneven distribution of data can lead to poorer results.

In [ ]:
_, inverse = np.unique(y_data, return_inverse=True)
ziffer_bincount = np.bincount(inverse)
fig = plt.figure(figsize=(40, 10))
fig.suptitle("Distribution of data")
plt.bar(np.arange (0, 100/10, 0.1), ziffer_bincount, width=0.09, align='center')
plt.ylabel('count')
plt.xlabel('digit class')
plt.tight_layout()
_ = plt.xticks(np.arange(0, 100/10, 0.1))


### Model Definition

The layout of the network ist a typcial CNN network with alternating 2x **Conv2D**, **BatchNormalization** and **MaxPool2D** layers. Finished after **flattening** with additional **Dense** layer.

**Dropout** between CNN layers are 0.2, between fully connected layers 0.4, but only the fully connected output layer.

* Shape of the input layer: (32, 32, 3)
* Shape of the output layer: (100) - classification 0.0 ... 9.9

In [ ]:
if (TFlite_Size == "s1"):
    model = model_ana_class100_s1(input_shape=(Input_Shape[0], Input_Shape[1], Input_Shape[2]), learning_rate=5e-4)
elif (TFlite_Size == "s2"):
    model = model_ana_class100_s2(input_shape=(Input_Shape[0], Input_Shape[1], Input_Shape[2]), learning_rate=5e-4)
else:
    raise ValueError(f"TFlite_Size: '{TFlite_Size}' is not supported.")


### Augmentation / Prepare Datasets

In [ ]:
# Parameter
Batch_Size = 8
Shift_Range = 1
Brightness_Range = 0.2
Zoom_Range = 0.05


def preprocessing(x):
    x = random_invert_image(x)
    x = random_white_balance(x)
    x = np.clip(x, 0.0, 255.0)
    return x.astype(np.float32)


# Training data
print("Training data")
datagen = ImageDataGenerator(width_shift_range=[-Shift_Range, Shift_Range], 
                             height_shift_range=[-Shift_Range, Shift_Range],
                             brightness_range=[1 - Brightness_Range, 1 + Brightness_Range],
                             zoom_range=[1 - Zoom_Range, 1 + Zoom_Range],
                             channel_shift_range=5,
                             shear_range=1,
                             preprocessing_function=preprocessing
                            )

train_iterator = datagen.flow(x_train, y_train, batch_size=Batch_Size)
plot_dataset_analog(train_iterator)

if (Validation_Percentage > 0):
    # Validation data
    datagen_val = ImageDataGenerator() # No augmentation for validation
    validation_iterator = datagen_val.flow(x_test, y_test, batch_size=Batch_Size)
    print("  ")
    print("Validation data")
    plot_dataset_analog(validation_iterator, rows=3)


### Training
 
* Train the model using augmented training data and unaltered validation data
* Visualize training and validation performance over epochs (e.g., loss and accuracy curves)
* Avoid significant overfitting: The validation performance should not deviate heavily from training due to augmentation being applied only to the training data
* Best-performing model automatically selected, not necessarily the final one. The optimal model is often found within the last ~30-40 epochs

In [ ]:
# Parameter
Epochs = 600


reduce_lr = ReduceLROnPlateau(
    monitor='val_loss', factor=0.9, patience=10, min_lr=1e-5, verbose=0
)

early_stopping = EarlyStopping(
    monitor='val_loss', mode='min', patience=40, restore_best_weights=True, verbose=0
)

if (Validation_Percentage > 0):
    history = model.fit(
        train_iterator, validation_data = validation_iterator, epochs = Epochs, 
        callbacks=[reduce_lr, early_stopping], verbose=0
    )
else:
    history = model.fit(
        train_iterator, epochs = Epochs, 
        callbacks=[reduce_lr, early_stopping], verbose=0
    )


In [ ]:
plt.semilogy(history.history['loss'])

if (Validation_Percentage > 0):
    plt.semilogy(history.history['val_loss'])

plt.title('model loss')
plt.ylabel('loss')
plt.xlabel('epoch')
plt.legend(['training','validation'], loc='upper left')
plt.grid(True)
plt.show()


### Model verification

* The following code uses the trained model to check the deviation for each picture (train + validation).
* The accepted_deviation can be used to get the accuracy with allowed differences (for instance +/- 0.1)
* The first (max) 49 false predicted images will be shown
* A csv-file with all false predicted images will be created. It can be used for relabeling with this tool: https://github.com/haverland/collectmeteranalog

In [ ]:
# Parameters
accepted_deviation = 0.1


# Predict
classes = model.predict(x_data.astype(np.float32))
predictions = np.argmax(classes, axis=1).reshape(-1)/10

# Wrap: 9.9 <> 0 = 0.1 and 1.1 <> 1.2 = 0.1
deviation_val = np.minimum(np.abs(predictions - y_data), np.abs(predictions - (10 - y_data)))

# Round values
predicted_val = np.round(predictions, 1)
expected_val = np.round(y_data, 1)
deviation_val = np.round(deviation_val, 1)

# Gater false predicted elements larger than configured delta
false_predicted_mask = deviation_val > accepted_deviation

false_predicted_dev_values = deviation_val[false_predicted_mask]
false_predicted_expected_values = expected_val[false_predicted_mask]
false_predicted_pred_values = predicted_val[false_predicted_mask]
false_predicted_images = x_data[false_predicted_mask]
#false_predicted_files = [f for f in f_data[false_predicted_mask]]
false_predicted_files = [Path(f).name for i, f in enumerate(f_data) if false_predicted_mask[i]]
false_predicted_plt_labels = [ 
    "Expected: " + str(y1) + "\n Predicted: " + str(p) + "\n" + str(Path(f).name[:-4][:20])
    for y1, p, f in zip(expected_val[false_predicted_mask], predicted_val[false_predicted_mask], f_data[false_predicted_mask])
]

# Sort by deviation (largest first)
sorted_indices = np.argsort(false_predicted_dev_values)[::-1]  # This sorts in descending order

false_predicted_dev_values = false_predicted_dev_values[sorted_indices]
false_predicted_expected_values = false_predicted_expected_values[sorted_indices]
false_predicted_pred_values = false_predicted_pred_values[sorted_indices]
false_predicted_images = false_predicted_images[sorted_indices]
false_predicted_plt_labels = [false_predicted_plt_labels[i] for i in sorted_indices]
false_predicted_files = [false_predicted_files[i] for i in sorted_indices]


In [ ]:
# Plot False Predicted Divergation
print(f"Accuracy: {(1 - len(false_predicted_dev_values) / len(y_data)) * 100.0:.2f} % (Images: {len(y_data)} | False Predicted: {len(false_predicted_dev_values)}) | Accepted Deviation: {accepted_deviation}")

title = f"False Predicted Divergation  |  Images: {len(y_data)}\nAccuracy: {(1 - len(false_predicted_dev_values) / len(y_data)) * 100.0:.2f} % (False Predicted: {len(false_predicted_dev_values)}) With Accepted Deviation: {accepted_deviation}"
_ = plot_divergence(np.bincount(np.array(np.round((np.abs(false_predicted_dev_values) * 10) % 51).astype(int)), minlength=51), title)


In [ ]:
# Plot the dataset of false predictions (Use first 49 entries)
print("False Predictions (Sorted by highest deviation, max. 49 images)")
plot_dataset_analog_result(false_predicted_images, false_predicted_plt_labels, columns=7, rows=7, figsize=(18,18))


In [ ]:
# Save false predicted image list to CSV
# The csv file can be further processed with the tool [collectmeteranalog](https://github.com/haverland/collectmeteranalog) to evaluate or adjust labels

csv_dir = f"{Output_Dir}/training_details/{TFlite_MainType}_{TFlite_Version}_{TFlite_Size}/"
Path(csv_dir).mkdir(parents=True, exist_ok=True)  # Create csv folder if it doesn't exist

false_predicted_df = pd.DataFrame(list(zip(false_predicted_files, false_predicted_pred_values, false_predicted_expected_values, 
                                  false_predicted_dev_values)), columns=["File", "Predicted", "Expected", "Deviation"])
false_predicted_df.to_csv(f"{csv_dir}/{TFlite_MainType}_{TFlite_Version}_{TFlite_Size}_false_predicted.csv", index=True, index_label="Index")


###############################################################################################################################################################################
# Save false predicted image list to CSV (--> LEGACY File Syntax)
# The csv file can be further processed with the tool [collectmeteranalog](https://github.com/haverland/collectmeteranalog) to evaluate or adjust labels

#false_predicted_df = pd.DataFrame(false_predicted_files)
#false_predicted_df.to_csv(f"{csv_dir}/{TFlite_MainType}_{TFlite_Version}_{TFlite_Size}_false_predicted.csv", index=True)


### Save the model

* Save the model to the file with the "tflite" file format
* quantize the model and store it as _q.tflite

In [ ]:
FileName = f"{Output_Dir}/{TFlite_MainType}_{TFlite_Version}_{TFlite_Size}.tflite"

# TensorFlow Lite conversion
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

# Save the converted model
with open(FileName, "wb") as f:
    f.write(tflite_model)

print(f"Model saved successfully. File: {FileName}")
print(f"File size: {Path(FileName).stat().st_size} bytes")


In [ ]:
FileName = f"{Output_Dir}/{TFlite_MainType}_{TFlite_Version}_{TFlite_Size}_q.tflite"

# Representative dataset function
def representative_dataset():
    for n in range(x_data[0].shape[0]):
        data = np.expand_dims(x_data[n], axis=0)
        yield [data.astype(np.float32)]

# TensorFlow Lite conversion with optimizations
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_dataset
converter._experimental_disable_per_channel_quantization_for_dense_layers = True
tflite_quant_model = converter.convert()

# Save the converted model to the specified file
with open(FileName, "wb") as f:
    f.write(tflite_quant_model)

print(f"Model saved successfully. File: {FileName}")
print(f"File size: {Path(FileName).stat().st_size} bytes")
